# Part 6B: Thiết kế Giao diện Người dùng & Bộ Mô phỏng Rủi ro Chuyến bay (Web Frontend UI & Interactive Simulator)
**Thành viên phụ trách:** Thành viên 7 | **Nhánh Git:** `feature/web-frontend` | **Mã nguồn:** `web/templates/index.html`, `web/static/style.css`

---
Notebook này trình bày quy trình thiết kế, kiểm định hợp đồng giao diện (DOM & CSS Contract) và đánh giá trải nghiệm người dùng cho nền tảng **FlightRisk VN**:

| # | Hạng mục công việc | Kiểm định đi kèm |
| :-: | :--- | :--- |
| 1 | Thiết kế cấu trúc HTML5 ngữ nghĩa (`web/templates/index.html`) | Đủ 4 Tab điều hướng & 11 trường đầu vào hợp lệ |
| 2 | Kiểm định Hợp đồng DOM với Client JS (`web/static/app.js` của TV8) | 100% ID (`#predict-form`, `#res-risk-level`, ...) khớp chính xác |
| 3 | Kiểm định Ràng buộc Chống Rò rỉ trên Giao diện (UI Zero-Leakage) | Không có ô nhập nào yêu cầu `actual_time`, `delay_min`, `status` |
| 4 | Thiết kế Thanh trượt Khí tượng & Tải sân bay khớp ngưỡng Feature Engineering | Miền giá trị mưa `[0, 25]`, gió `[5, 80]`, tầm nhìn `[500, 10000]`, tải `[5, 50]` |
| 5 | Hệ thống Thiết kế CSS (`web/static/style.css`) & Bảng màu Rủi ro 3 cấp | Đủ biến CSS (`:root`), class `.risk-low/.risk-medium/.risk-high` & Responsive `@media` |
| 6 | Đối chiếu 3 kịch bản mô phỏng chuẩn (UAT Presets: Trời quang / Gió mạnh / Mưa dông) | Màu và xác suất phụ thuộc kết quả thực tế từ API |

> **Quy tắc làm việc nhóm:** Notebook đã được xóa output (`Clear All Outputs`) trước khi commit lên Git theo đúng quy định tại `README.md`.

In [ ]:
import os
import re
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

ROOT = Path.cwd()
if not (ROOT / "config.py").exists() and (ROOT.parent / "config.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

HTML_PATH = ROOT / "web" / "templates" / "index.html"
CSS_PATH = ROOT / "web" / "static" / "style.css"
JS_PATH = ROOT / "web" / "static" / "app.js"

assert HTML_PATH.exists(), f"Không tìm thấy {HTML_PATH}"
assert CSS_PATH.exists(), f"Không tìm thấy {CSS_PATH}"
print(f"Đã nạp thành công:\n - HTML: {HTML_PATH} ({HTML_PATH.stat().st_size:,} bytes)\n - CSS : {CSS_PATH} ({CSS_PATH.stat().st_size:,} bytes)")

## 1. Kiểm định Hợp đồng DOM (DOM Contract Verification) giữa TV7 (HTML/CSS) và TV8 (`app.js`)
Để mã JavaScript xử lý AJAX (`web/static/app.js` do Thành viên 8 phụ trách) hoạt động không lỗi, tệp `web/templates/index.html` của Thành viên 7 bắt buộc phải cung cấp đầy đủ các phần tử DOM với `id` và `data-tab` chuẩn xác.

In [ ]:
soup = BeautifulSoup(HTML_PATH.read_text(encoding="utf-8"), "html.parser")

REQUIRED_IDS = [
    # Form & Inputs (12 IDs)
    "predict-form", "airline", "airport", "origin", "destination",
    "direction", "route-type", "flight-date", "sched-time",
    "precip", "wind", "vis", "weather-code", "load",
    # Live Slider Readouts (4 IDs)
    "precip-val", "wind-val", "vis-val", "load-val",
    # Result Risk Card & Multi-class Probability Bars (12 IDs)
    "predict-result", "res-risk-level", "res-prediction-text", "res-risk-score",
    "prob-ontime", "prob-ontime-text", "prob-delayed", "prob-delayed-text",
    "prob-cancelled", "prob-cancelled-text",
    "res-cluster-name", "res-cluster-rate", "res-rules-list",
    # 4 Tab Sections & Lazy-loaded Containers (7 IDs)
    "simulator-tab", "models-tab", "clusters-tab", "rules-tab",
    "metrics-content", "clusters-content", "rules-content",
]

rows = []
for elem_id in REQUIRED_IDS:
    el = soup.find(id=elem_id)
    rows.append({
        "DOM ID": f"#{elem_id}",
        "Thẻ HTML": f"<{el.name}>" if el else "MISSING",
        "Trạng thái": "PASS" if el is not None else "FAIL",
    })

df_dom = pd.DataFrame(rows)
display(df_dom)
assert (df_dom["Trạng thái"] == "PASS").all(), "Có phần tử DOM bắt buộc bị thiếu!"

## 2. Phân tích Các Trường Đầu vào & Ngưỡng Thanh trượt (Sliders vs Feature Engineering)
Mọi thanh trượt trên giao diện được thiết kế khớp trực tiếp với các ngưỡng rời rạc hóa trong `features/features.py` (Part 3) và `pipeline/transformation.py` (Part 2).

In [ ]:
sliders_info = []
for sid, desc, fe_rule in [
    ("precip", "Lượng mưa (mm)", "is_rain > 0.1mm | is_heavy_rain > 2.5mm | max scale = 10mm"),
    ("wind", "Gió giật 10m (km/h)", "is_high_wind > 30km/h | is_severe_wind > 50km/h"),
    ("vis", "Tầm nhìn xa (m)", "is_low_vis < 5,000m | chuẩn = 10,000m"),
    ("load", "Tải sân bay (chuyến/giờ)", "is_heavy_traffic > 28 chuyến/giờ"),
]:
    el = soup.find(id=sid)
    sliders_info.append({
        "Slider ID": f"#{sid}",
        "Thông số": desc,
        "Min": el.get("min"),
        "Max": el.get("max"),
        "Step": el.get("step"),
        "Default": el.get("value"),
        "Ngưỡng Feature Engineering (Part 3)": fe_rule,
    })

display(pd.DataFrame(sliders_info))

## 3. Kiểm định Hệ thống Bảng màu (Color Palette) & Responsive CSS (`web/static/style.css`)
Giao diện sử dụng hệ thống CSS Custom Properties (`:root`) để đồng bộ nhận diện thương hiệu hàng không và 3 cấp độ cảnh báo rủi ro (`LOW`, `MEDIUM`, `HIGH`).

In [ ]:
css_text = CSS_PATH.read_text(encoding="utf-8")
vars_found = re.findall(r"(--[a-z0-9-]+)\s*:\s*([^;]+);", css_text)
df_css_vars = pd.DataFrame(vars_found, columns=["Biến CSS (Token)", "Giá trị"]).head(18)
display(df_css_vars)

# Trực quan hóa bảng màu nhận diện thương hiệu và cảnh báo rủi ro
palette = [
    ("Primary Navy", "#1e3a8a"),
    ("Primary Blue", "#2563eb"),
    ("Secondary Slate", "#0f172a"),
    ("LOW Risk (On-Time)", "#10b981"),
    ("MEDIUM Risk (Caution)", "#f59e0b"),
    ("HIGH Risk (Delay/Cancel)", "#ef4444"),
]
fig, ax = plt.subplots(figsize=(9, 1.8))
for idx, (label, hex_col) in enumerate(palette):
    ax.barh(0, 1, left=idx, color=hex_col, edgecolor="white", height=0.8)
    ax.text(idx + 0.5, 0, f"{label}\n{hex_col}", ha="center", va="center", color="white", fontweight="bold", fontsize=8.5)
ax.set_xlim(0, len(palette))
ax.axis("off")
plt.title("Bảng Màu Nhận Diện Thương Hiệu & Cảnh Báo Rủi Ro (FlightRisk VN UI)", fontweight="bold", pad=10)
plt.tight_layout()
plt.show()

## 4. Tổng kết đóng góp vào Báo cáo (Chương 6 — Mục 6.2: Thiết kế Giao diện Người dùng)
- **Kiến trúc Single-Page Dashboard 4 Tabs:** Tích hợp trọn vẹn luồng suy luận thời gian thực (Tab 1) với kết quả đánh giá Supervised ML (Tab 2), Hồ sơ phân cụm K-Means & Isolation Forest (Tab 3), và Tập luật kết hợp FP-Growth (Tab 4).
- **Trải nghiệm Mô phỏng Tương tác (Interactive Simulator):**
  - Hỗ trợ 3 nút **UAT Preset 1-Click** (Trời quang / Gió mạnh / Mưa dông) giúp trình diễn nhanh sự thay đổi tức thời của Risk Card và Probability Bars.
  - Tích hợp thẻ **Flight Boarding Pass** tự động nhận diện đường bay *Tam Giác Vàng* (`SGN-HAN-DAD`), *Nội địa* hoặc *Quốc tế*, kèm nút đảo chiều sân bay `⇄`.
  - Toàn bộ thanh trượt hiển thị mốc chuẩn hóa theo đúng công thức toán học của `weather_hazard_score` và `airport_load`.